The objective is to identify unusual borrower profiles using:
1. Density-based anomaly detection: DBSCAN noise observations (label = -1).
2. Feature-level analysis: Identify borrowers with unusually high or low revenue, loan amounts, FICO scores and DTI.
3. Default comparison: Examine the historical default rate among detected anomalies, using Default only after anomaly detection.
We will use a 5,000-borrower sample for computational efficiency, consistent with your previous clustering analysis.

Step 1 — Load dataset and prepare features

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(
    r"D:\HCL_PROJECTS\Credit Portfolio Segmentation and Risk Pattern Analysis Project-3\Pickle\cleaned_credit_portfolio.pkl"
)

df = pd.read_pickle(DATA_PATH)      # not read_csv

print("Dataset shape:", df.shape)
display(df.head())

# Check dataset
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# Check required columns
required_columns = [
    'revenue',
    'loan_amnt',
    'fico_n',
    'dti_capped',
    'revenue_log',
    'emp_length',
    'purpose',
    'home_ownership_n',
    'Default'
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

print("\nMissing required columns:", missing_columns)

# Create loan amount log if not already present
if 'loan_amnt_log' not in df.columns:
    df['loan_amnt_log'] = np.log1p(df['loan_amnt'])

print("\nMissing values in required columns:")
print(df[required_columns + ['loan_amnt_log']].isnull().sum())

Dataset shape: (1347681, 15)


,id,issue_d,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,Default,revenue_log,dti_capped,dti_extreme
0,68407277,Dec-2015,55000.0,5.91,3600,677.0,1,10+ years,debt_consolidation,MORTGAGE,PA,0,10.915107,5.91,0
1,68355089,Dec-2015,65000.0,16.06,24700,717.0,1,10+ years,small_business,MORTGAGE,SD,0,11.082158,16.06,0
2,68341763,Dec-2015,71000.0,13.85,20000,697.0,1,10+ years,home_improvement,MORTGAGE,IL,0,11.170449,13.85,0
3,68476807,Dec-2015,104433.0,25.37,10400,697.0,1,3 years,major_purchase,MORTGAGE,PA,0,11.556311,25.37,0
4,68426831,Dec-2015,34000.0,10.20,11950,692.0,1,4 years,debt_consolidation,RENT,GA,0,10.434145,10.20,0


Dataset shape: (1347681, 15)

Columns:
['id', 'issue_d', 'revenue', 'dti_n', 'loan_amnt', 'fico_n', 'experience_c', 'emp_length', 'purpose', 'home_ownership_n', 'addr_state', 'Default', 'revenue_log', 'dti_capped', 'dti_extreme']

Missing required columns: []

Missing values in required columns:
revenue             0
loan_amnt           0
fico_n              0
dti_capped          0
revenue_log         0
emp_length          0
purpose             0
home_ownership_n    0
Default             0
loan_amnt_log       0
dtype: int64


Step 2 — Select features and create a representative sample
For anomaly detection, we will use the same feature groups as your earlier clustering analysis:
- Numerical: revenue, loan amount, FICO and DTI (transformed).
- Categorical: employment length, loan purpose and home ownership.
- Default, id and addr_state will be excluded from the detection inputs.

In [3]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Numerical features
numeric_features = [
    'revenue_log',
    'loan_amnt_log',
    'fico_n',
    'dti_capped'
]

# Categorical features
categorical_features = [
    'emp_length',
    'purpose',
    'home_ownership_n'
]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Transform features (Default excluded)
X_scaled = preprocessor.fit_transform(df)

print("Transformed feature matrix shape:", X_scaled.shape)
print("Matrix type:", type(X_scaled))

Transformed feature matrix shape: (1347681, 34)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>


Create a reproducible 5,000-borrower sample

In [4]:
rng = np.random.default_rng(42)

sample_size = 5000

sample_indices = rng.choice(
    len(df),
    size=sample_size,
    replace=False
)

X_anomaly_sample = X_scaled[sample_indices]

print("Anomaly sample shape:", X_anomaly_sample.shape)

Anomaly sample shape: (5000, 34)


Step 3 — PCA dimensionality reduction

In [5]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Convert sampled sparse matrix to dense
X_anomaly_dense = X_anomaly_sample.toarray()

# Apply PCA
pca_anomaly = PCA(
    n_components=10,
    random_state=42
)

X_anomaly_pca = pca_anomaly.fit_transform(
    X_anomaly_dense
)

print("PCA shape:", X_anomaly_pca.shape)

print(
    "Total explained variance:",
    round(pca_anomaly.explained_variance_ratio_.sum() * 100, 2),
    "%"
)

# Standardize PCA components
pca_anomaly_scaler = StandardScaler()

X_anomaly_std = pca_anomaly_scaler.fit_transform(
    X_anomaly_pca
)

print("Standardized PCA shape:", X_anomaly_std.shape)
print("Missing values:", np.isnan(X_anomaly_std).sum())

PCA shape: (5000, 10)
Total explained variance: 89.18 %
Standardized PCA shape: (5000, 10)
Missing values: 0


Step 4 — DBSCAN anomaly detection

In [6]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

eps_values = [1.5, 1.8, 2.1, 2.4, 2.6]

dbscan_results = []

for eps in eps_values:

    model = DBSCAN(
        eps=eps,
        min_samples=10,
        metric='euclidean'
    )

    labels = model.fit_predict(X_anomaly_std)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    noise_count = np.sum(labels == -1)

    noise_percentage = noise_count / len(labels) * 100

    dbscan_results.append({
        'Epsilon': eps,
        'Clusters': n_clusters,
        'Noise_Count': noise_count,
        'Noise_Percentage': round(noise_percentage, 2)
    })

dbscan_results_df = pd.DataFrame(dbscan_results)

print("DBSCAN ANOMALY DETECTION RESULTS")
print(dbscan_results_df)

DBSCAN ANOMALY DETECTION RESULTS
   Epsilon  Clusters  Noise_Count  Noise_Percentage
0      1.5        22          598             11.96
1      1.8        24          263              5.26
2      2.1        13           96              1.92
3      2.4         3           50              1.00
4      2.6         3           31              0.62


Step 4 – DBSCAN parameter evaluation
Your results show that increasing eps reduces the number of observations classified as noise.
Epsilon	Clusters	Noise observations	Noise %
1.5	22	598	11.96%
1.8	24	263	5.26%
2.1	13	96	1.92%
2.4	3	50	1.00%
2.6	3	31	0.62%

Recommended starting choice: eps=1.8, min_samples=10.

It identifies 263 potentially unusual borrowers (5.26%) while retaining 24 density-based clusters. At eps=1.5, nearly 12% are labelled noise, which may be too aggressive. At higher epsilon values, very few observations are flagged.

This is a practical starting point, not proof that these borrowers are genuinely anomalous. We will inspect their profiles next.

Step 5 – Identify unusual borrower profiles
Now fit DBSCAN using the selected parameters and separate the noise observations (-1) from the remaining borrowers.

In [7]:
from sklearn.cluster import DBSCAN

# Fit final DBSCAN model
dbscan_final = DBSCAN(
    eps=1.8,
    min_samples=10,
    metric='euclidean'
)

anomaly_labels = dbscan_final.fit_predict(X_anomaly_std)

# Create sample profile
anomaly_df = df.iloc[sample_indices].copy().reset_index(drop=True)

anomaly_df['DBSCAN_Cluster'] = anomaly_labels

# Identify noise observations
anomaly_df['Anomaly_Status'] = np.where(
    anomaly_df['DBSCAN_Cluster'] == -1,
    'Potential Anomaly',
    'Normal'
)

print("ANOMALY DISTRIBUTION")
print(anomaly_df['Anomaly_Status'].value_counts())

print("\nANOMALY PERCENTAGE")
print(
    (anomaly_df['Anomaly_Status'].value_counts(normalize=True) * 100)
    .round(2)
)

ANOMALY DISTRIBUTION
Anomaly_Status
Normal               4737
Potential Anomaly     263
Name: count, dtype: int64

ANOMALY PERCENTAGE
Anomaly_Status
Normal               94.74
Potential Anomaly     5.26
Name: proportion, dtype: float64


Step 5 – Compare normal and potential anomaly profiles
Now investigate how the two groups differ in their financial characteristics.

# Refit DBSCAN using the selected parameters
from sklearn.cluster import DBSCAN

dbscan_final = DBSCAN(
    eps=1.8,
    min_samples=10,
    metric='euclidean'
)

final_labels = dbscan_final.fit_predict(X_anomaly_std)

# Create a sample-level analysis DataFrame
anomaly_df = df.iloc[sample_indices].copy().reset_index(drop=True)

anomaly_df['DBSCAN_Cluster'] = final_labels

anomaly_df['Anomaly_Status'] = np.where(
    final_labels == -1,
    'Potential Anomaly',
    'Normal'
)

# Compare financial characteristics
anomaly_profile = anomaly_df.groupby('Anomaly_Status').agg(
    Borrower_Count=('Default', 'count'),
    Average_Revenue=('revenue', 'mean'),
    Median_Revenue=('revenue', 'median'),
    Average_Loan_Amount=('loan_amnt', 'mean'),
    Average_FICO=('fico_n', 'mean'),
    Average_DTI=('dti_capped', 'mean'),
    Default_Count=('Default', 'sum'),
    Default_Rate=('Default', 'mean')
)

# Convert to percentage first, then round (rounding before x100 loses precision)
anomaly_profile['Default_Rate'] = anomaly_profile['Default_Rate'] * 100
anomaly_profile = anomaly_profile.round(2)

print("BORROWER PROFILE: NORMAL vs POTENTIAL ANOMALY")
print(anomaly_profile)

Step 5 – Interpretation
Your DBSCAN results reveal some interesting differences between normal borrowers and potential anomalies.
Financial indicator	Normal	Potential Anomaly
Borrowers	4,737	263
Average revenue	$75,732	$110,254
Median revenue	$65,000	$65,000
Average loan	$14,675	$12,010
Average FICO	696.16	729.32
Average DTI	18.24	18.90
Default rate	21.0%	17.0%

Key observations:
1. Higher average revenue: Potential anomalies have approximately 45.6% higher average revenue.
2. Median revenue is identical: Both groups have a median revenue of $65,000. This suggests that unusually high revenue observations may be influencing the anomaly group's average.
3. Higher credit scores: Potential anomalies have an average FICO score of 729.32, compared with 696.16 for normal borrowers.
4. Lower average loan amounts: Potential anomalies have smaller average loans.
5. Lower default rate: Potential anomalies have a 17.11% default rate, compared with 20.94% for normal borrowers.
Important conclusion: DBSCAN has identified unusual borrower profiles, but these observations are not necessarily financially risky. In this sample, potential anomalies actually have a lower default rate.

Step 6 – Feature-level anomaly analysis
Next, investigate whether individual financial features contain extreme values using the Interquartile Range (IQR) method.

In [8]:
# Feature-level outlier detection using IQR

features = [
    'revenue',
    'loan_amnt',
    'fico_n',
    'dti_capped'
]

iqr_results = []

for col in features:

    Q1 = anomaly_df[col].quantile(0.25)
    Q3 = anomaly_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (anomaly_df[col] < lower_bound) |
        (anomaly_df[col] > upper_bound)
    ).sum()

    iqr_results.append({
        'Feature': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Lower_Bound': round(lower_bound, 2),
        'Upper_Bound': round(upper_bound, 2),
        'Outlier_Count': outlier_count,
        'Outlier_Percentage': round(
            outlier_count / len(anomaly_df) * 100, 2
        )
    })

iqr_results_df = pd.DataFrame(iqr_results)

print("FEATURE-LEVEL OUTLIER ANALYSIS")
print(iqr_results_df)

FEATURE-LEVEL OUTLIER ANALYSIS
      Feature        Q1        Q3       IQR  Lower_Bound  Upper_Bound  \
0     revenue  47000.00  92625.00  45625.00    -21437.50    161062.50   
1   loan_amnt   8000.00  20000.00  12000.00    -10000.00     38000.00   
2      fico_n    672.00    712.00     40.00       612.00       772.00   
3  dti_capped     11.81     24.32     12.51        -6.96        43.08   

   Outlier_Count  Outlier_Percentage  
0            244                4.88  
1             23                0.46  
2            181                3.62  
3             17                0.34  


Step 6 – Interpretation of feature-level outliers
Your IQR analysis identifies the following unusual values among the 5,000 sampled borrowers:
Feature	Outlier count	Percentage
Revenue	244	4.88%
Loan amount	23	0.46%
FICO score	181	3.62%
DTI	17	0.34%

Key findings:
1. Revenue: Highest number of outliers (244). Some borrowers have unusually high or low reported revenue.
2. FICO: 181 outliers. Scores outside 612–772 are flagged by the IQR rule.
3. Loan amount: Only 23 outliers, indicating relatively few extreme loan amounts.
4. DTI: Only 17 outliers, indicating relatively few extreme DTI observations.
Important observation: DBSCAN identified 263 potential anomalies, while the individual IQR tests identified outliers independently for each feature. These groups do not necessarily contain the same borrowers.

Step 7 – Compare DBSCAN anomalies with feature-level outliers
This will help establish whether DBSCAN anomalies also have unusual individual financial features.

In [9]:
# Identify IQR outliers for each financial feature

features = ['revenue', 'loan_amnt', 'fico_n', 'dti_capped']

for col in features:

    Q1 = anomaly_df[col].quantile(0.25)
    Q3 = anomaly_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    anomaly_df[col + '_IQR_Outlier'] = (
        (anomaly_df[col] < lower) |
        (anomaly_df[col] > upper)
    )

# Count feature-level outliers within each DBSCAN group

comparison = anomaly_df.groupby('Anomaly_Status')[
    [col + '_IQR_Outlier' for col in features]
].sum()

print("FEATURE-LEVEL OUTLIERS BY DBSCAN STATUS")
print(comparison)

# Percentage comparison

comparison_pct = (
    anomaly_df.groupby('Anomaly_Status')[
        [col + '_IQR_Outlier' for col in features]
    ].mean() * 100
).round(2)

print("\nOUTLIER PERCENTAGE BY DBSCAN STATUS")
print(comparison_pct)

FEATURE-LEVEL OUTLIERS BY DBSCAN STATUS
                   revenue_IQR_Outlier  loan_amnt_IQR_Outlier  \
Anomaly_Status                                                  
Normal                             206                     16   
Potential Anomaly                   38                      7   

                   fico_n_IQR_Outlier  dti_capped_IQR_Outlier  
Anomaly_Status                                                 
Normal                            119                       5  
Potential Anomaly                  62                      12  

OUTLIER PERCENTAGE BY DBSCAN STATUS
                   revenue_IQR_Outlier  loan_amnt_IQR_Outlier  \
Anomaly_Status                                                  
Normal                            4.35                   0.34   
Potential Anomaly                14.45                   2.66   

                   fico_n_IQR_Outlier  dti_capped_IQR_Outlier  
Anomaly_Status                                                 
Normal           

Step 7 – Interpretation: DBSCAN vs feature-level outliers
Your results show that potential DBSCAN anomalies have a substantially higher concentration of individual feature-level outliers.
Feature	Normal (%)	Potential Anomaly (%)
Revenue	4.35	14.45
Loan amount	0.34	2.66
FICO	2.51	23.57
DTI	0.11	4.56

Key findings:
1. FICO: 23.57% of potential anomalies have unusual FICO values, compared with only 2.51% of normal borrowers.
2. Revenue: 14.45% of potential anomalies have unusual revenue values.
3. DTI: 4.56% of potential anomalies have unusual DTI values.
4. Loan amount: 2.66% of potential anomalies have unusual loan amounts.
This supports the usefulness of DBSCAN for identifying borrowers with unusual combinations of financial characteristics.

An important distinction: DBSCAN detects unusual combinations across multiple features, whereas IQR detects extreme values in individual features.

Step 8 – Anomaly detection summary
Let's generate a final summary table for your project report.

In [10]:
# Final anomaly detection summary

summary = pd.DataFrame({
    'Metric': [
        'Total Borrowers Analyzed',
        'Normal Borrowers',
        'Potential Anomalies',
        'Anomaly Percentage',
        'Revenue Outliers in Anomalies',
        'Loan Amount Outliers in Anomalies',
        'FICO Outliers in Anomalies',
        'DTI Outliers in Anomalies'
    ],

    'Value': [
        len(anomaly_df),
        (anomaly_df['Anomaly_Status'] == 'Normal').sum(),
        (anomaly_df['Anomaly_Status'] == 'Potential Anomaly').sum(),
        round(
            (anomaly_df['Anomaly_Status'] == 'Potential Anomaly').mean() * 100, 2
        ),
        anomaly_df.loc[
            anomaly_df['Anomaly_Status'] == 'Potential Anomaly',
            'revenue_IQR_Outlier'
        ].sum(),
        anomaly_df.loc[
            anomaly_df['Anomaly_Status'] == 'Potential Anomaly',
            'loan_amnt_IQR_Outlier'
        ].sum(),
        anomaly_df.loc[
            anomaly_df['Anomaly_Status'] == 'Potential Anomaly',
            'fico_n_IQR_Outlier'
        ].sum(),
        anomaly_df.loc[
            anomaly_df['Anomaly_Status'] == 'Potential Anomaly',
            'dti_capped_IQR_Outlier'
        ].sum()
    ]
})

print("FINAL ANOMALY DETECTION SUMMARY")
print(summary)

FINAL ANOMALY DETECTION SUMMARY
                              Metric    Value
0           Total Borrowers Analyzed  5000.00
1                   Normal Borrowers  4737.00
2                Potential Anomalies   263.00
3                 Anomaly Percentage     5.26
4      Revenue Outliers in Anomalies    38.00
5  Loan Amount Outliers in Anomalies     7.00
6         FICO Outliers in Anomalies    62.00
7          DTI Outliers in Anomalies    12.00


Your anomaly detection analysis is complete. Below is the consolidated interpretation suitable for your project report.
1. Final anomaly detection results
Metric	Result
Total borrowers analyzed	5,000
Normal borrowers	4,737
Potential anomalies	263
Anomaly percentage	5.26%
Revenue outliers among anomalies	38
Loan amount outliers among anomalies	7
FICO outliers among anomalies	62
DTI outliers among anomalies	12

Selected DBSCAN parameters:
- Epsilon (eps): 1.8
- Minimum samples: 10
- Distance metric: Euclidean
- PCA components: 10
- Explained variance: 89.18%
2. Financial profile comparison
Indicator	Normal	Potential Anomaly
Average revenue	$75,732	$110,254
Median revenue	$65,000	$65,000
Average loan amount	$14,675	$12,010
Average FICO	696.16	729.32
Average DTI	18.24	18.90
Default rate	20.94%	17.11%

Potential anomalies generally have higher average revenue and FICO scores, smaller average loan amounts, and a lower observed default rate.
3. Feature-level analysis
The IQR method identified revenue as the feature with the highest overall outlier count, followed by FICO.

Within the DBSCAN potential anomaly group:
- 14.45% had unusual revenue values.
- 23.57% had unusual FICO values.
- 4.56% had unusual DTI values.
- 2.66% had unusual loan amounts.
This indicates that potential anomalies frequently exhibit unusual combinations of financial characteristics rather than simply extreme loan amounts.
4. Project report – Final conclusion
Anomaly Detection
Anomaly detection was performed using DBSCAN clustering and feature-level outlier analysis to investigate unusual borrower profiles.

A random sample of 5,000 borrowers was selected from the original dataset. The numerical and categorical features were standardized and encoded, followed by PCA dimensionality reduction. Ten principal components retained 89.18% of the total variance.

DBSCAN was evaluated using different epsilon values with min_samples = 10. Based on the resulting cluster structure and noise percentage, an epsilon value of 1.8 was selected.

The final model identified 263 potential anomalies (5.26%) and 4,737 normal observations (94.74%).

A comparison of financial characteristics showed that potential anomalies had higher average revenue ($110,254), higher average FICO scores (729.32), and lower average loan amounts ($12,010) than normal borrowers. Their observed default rate was also lower (17.11% compared with 20.94% for normal borrowers).

Feature-level outlier detection using the IQR method revealed that FICO and revenue were particularly prominent among the unusual profiles identified by DBSCAN. In the potential anomaly group, 23.57% had unusual FICO values and 14.45% had unusual revenue values.

Conclusion: DBSCAN successfully identified a subset of borrowers with unusual multivariate financial profiles. The feature-level analysis provided additional insight into the characteristics contributing to these unusual observations. However, potential anomalies should not automatically be considered fraudulent, erroneous, or high-risk borrowers. Further investigation would be required before taking any action based on these findings.




5. Important methodological note
The 38 revenue, 7 loan, 62 FICO and 12 DTI outliers are feature-specific counts. They may overlap, so they should not be added together to calculate the total number of unique anomalies.

Also, this analysis is based on a 5,000-borrower sample, not the entire 1.35-million-row dataset.